# Phase 3 — SFT warm-start of AV + AR on Colab T4

**Prereqs**
1. Upload `guppylm-nla-bundle.tar.gz` (~114 MB) to your Google Drive `/MyDrive/`. (One-time. Holds the Phase 1+2 data + GuppyLM checkpoint.)
2. Runtime → Change runtime type → **T4 GPU**.

**Code source**: this notebook clones `https://github.com/suvojit-0x55aa/guppylm-nla` fresh on each run, so any hot-fix in the repo lands here via `!git pull` (or by re-running cell 5).

**Outputs** (saved back to Drive `/MyDrive/guppylm-nla-out/`)
- `history_text.json`, `history_lens.json` — per-variant training curves + final FVE
- `MANIFEST_phase3.json` — provenance + decision banner
- `phase3_ckpts.tar.gz` — final LoRA + projection/head checkpoints

## 1. Setup (pip install + mount Drive + untar)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U 'torchao>=0.16' 'transformers>=4.43' 'accelerate>=0.30' \
    'peft>=0.12' 'datasets>=2.20' 'bitsandbytes>=0.43' 'numpy>=1.24' \
    'tqdm>=4.65' 'tokenizers>=0.19' 'huggingface_hub>=0.20' 'pyarrow>=14' \
    'pytest>=7.4' 'pytest-asyncio>=0.21' 'openai>=1.40'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Scope all Drive-side artifacts under /MyDrive/guppylm-nla/.
# - hf_cache/   ← Qwen 2.5 3B (~6 GB compressed) cached so future sessions skip the download
# - out/        ← Phase 3 results (history, manifest, plot, ckpts) get written here
import os
PROJECT_ROOT = '/content/drive/MyDrive/guppylm-nla'
os.environ['HF_HOME'] = os.path.join(PROJECT_ROOT, 'hf_cache')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, 'out'), exist_ok=True)
print('HF_HOME    =', os.environ['HF_HOME'])
print('OUT folder =', os.path.join(PROJECT_ROOT, 'out'))

In [ ]:
import os, shutil, subprocess
BUNDLE = '/content/drive/MyDrive/guppylm-nla-bundle.tar.gz'
WORK = '/content/guppylm-infer'
REPO = 'https://github.com/suvojit-0x55aa/guppylm-nla.git'

# Move out of WORK before deleting it (prior runs may have %cd'd into it).
os.chdir('/content')

if os.path.exists(WORK):
    shutil.rmtree(WORK)
subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True, cwd='/content')

# Bundle holds data/ + checkpoints/ (~85MB). Untar to a staging dir and copy
# only those two trees into the cloned repo — code comes from git.
STAGE = '/content/_bundle_stage'
if os.path.exists(STAGE):
    shutil.rmtree(STAGE)
os.makedirs(STAGE)
subprocess.run(['tar', 'xzf', BUNDLE, '-C', STAGE], check=True, cwd='/content')
for sub in ('data', 'checkpoints'):
    src = os.path.join(STAGE, sub)
    dst = os.path.join(WORK, sub)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'synced {sub}/')

%cd {WORK}
!git log --oneline -3
!ls -la data/ checkpoints/

In [ ]:
# Hot-fix shortcut: pull only code (no re-untar). Run this if a fix lands
# in the repo while the runtime is still alive.
%cd /content/guppylm-infer
!git pull --ff-only
!git log --oneline -3

## 2. Sanity tests (no GPU; ~10s)

In [ ]:
!python -m pytest tests/test_av_ar.py -v

## 3. Smoke run (~3 min) — verifies the full pipeline on the actual GPU

In [ ]:
!python scripts/03_warmstart.py --variant text \
    --max-steps 50 --min-steps 0 --time-budget-min 5 \
    --batch 0 --eval-batch 0 \
    --fve-eval-size 16 --final-fve-eval-size 16 \
    --ckpt-root /content/drive/MyDrive/guppylm-nla/ckpts/phase3_smoke \
    --history-root /content/drive/MyDrive/guppylm-nla/out/smoke \
    --manifest /content/drive/MyDrive/guppylm-nla/out/smoke/MANIFEST_phase3_smoke.json

In [ ]:
import json
smoke = json.loads(open('/content/drive/MyDrive/guppylm-nla/out/smoke/history_text.json').read())
print('smoke FVE:', smoke['final_fve'])
print('smoke MSE:', smoke['final_mse'])
print('AV stop:', smoke['av']['stop_reason'], 'steps:', smoke['av']['final_step'])
print('AR stop:', smoke['ar']['stop_reason'], 'steps:', smoke['ar']['final_step'])
if smoke['samples']:
    print('\nfirst sample:'); print(smoke['samples'][0])

## 4. Full text variant (~60–180 min, convergence- or budget-bounded)

In [ ]:
!python scripts/03_warmstart.py --variant text --time-budget-min 180 \
    --batch 0 --eval-batch 0 \
    --ckpt-root /content/drive/MyDrive/guppylm-nla/ckpts/phase3 \
    --history-root /content/drive/MyDrive/guppylm-nla/out \
    --manifest /content/drive/MyDrive/guppylm-nla/out/MANIFEST_phase3.json

## 5. Full lens variant (~60–180 min)

In [ ]:
!python scripts/03_warmstart.py --variant lens --time-budget-min 180 \
    --batch 0 --eval-batch 0 \
    --ckpt-root /content/drive/MyDrive/guppylm-nla/ckpts/phase3 \
    --history-root /content/drive/MyDrive/guppylm-nla/out \
    --manifest /content/drive/MyDrive/guppylm-nla/out/MANIFEST_phase3.json

## 6. Plot FVE curves and apply decision rule

In [ ]:
import json, os
import matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/guppylm-nla/out'
ht = json.load(open(f'{OUT}/history_text.json'))
hl = json.load(open(f'{OUT}/history_lens.json'))

def fve_curve(h):
    return [(e['step'], e['fve']) for e in h['ar']['history'] if 'fve' in e]

fig, ax = plt.subplots(figsize=(10, 5))
for h, label, color in [(ht, 'text', 'C0'), (hl, 'lens', 'C1')]:
    pts = fve_curve(h)
    if pts:
        xs, ys = zip(*pts)
        ax.plot(xs, ys, color=color, label=f'{label} (final={h["final_fve"]:.3f})', marker='o')
ax.axhline(0.10, color='red', linestyle='--', alpha=0.5, label='hard-fail floor (0.10)')
ax.axhline(0.25, color='green', linestyle='--', alpha=0.5, label='Phase 4 threshold (0.25)')
ax.set_xlabel('AR training step'); ax.set_ylabel('FVE')
ax.set_title('Joint AV→AR FVE on held-out 500')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{OUT}/fve_curves.png', dpi=120)
plt.show()

In [ ]:
!cat /content/drive/MyDrive/guppylm-nla/out/MANIFEST_phase3.json

## 7. Save artifacts back to Drive

In [ ]:
# Outputs and checkpoints write directly to Drive during training (see cells 13/15).
# Confirm what's there:
!ls -la /content/drive/MyDrive/guppylm-nla/out/
!ls -la /content/drive/MyDrive/guppylm-nla/ckpts/phase3/